1. Create a scatterplot based on the dataset using two variables, namely Avg_Credit_Limit and Total_credit_card. Can the distribution of the data be considered linear?
2. Construct a clustering model using the K-Means algorithm. Determine the optimal number k using elbow method. Supplement your analysis with line chart and provide an analytical rationale for why you selected that value.
3. Depict the clustering results from the K-Means model using a scatter plot.
4. Construct a clustering model using DBSCAN with all usable variables.
5. Determine the number k using elbow method. Supplement your analysis with line chart and provide an analytical rationale for why you selected that value.
6. Determine the value of the silhouette score and the DBI of the DBSCAN model.

#### Question 1 - Scatterplot and linearity


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("CC_Customer_Dataset.csv")

plt.figure(figsize=(8, 5))
plt.scatter(
    df["Avg_Credit_Limit"],
    df["Total_Credit_Cards"],
    alpha=0.7,
)
plt.title("Average Credit Limit vs Total Credit Cards")
plt.xlabel("Average Credit Limit")
plt.ylabel("Total Credit Cards")
plt.tight_layout()
plt.show()

correlation = df["Avg_Credit_Limit"].corr(df["Total_Credit_Cards"])
print(f"Pearson correlation: {correlation:.3f}")
print(
    "Conclusion: the distribution is not strongly linear; the points form "
    "several groups rather than one straight-line pattern."
)

#### Question 2 - K-Means and the elbow method

The identifier columns (`Sl_No` and `Customer Key`) are excluded. The remaining columns are customer behavior and credit-limit variables, so they are the usable features for clustering.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

feature_columns = [
    "Avg_Credit_Limit",
    "Total_Credit_Cards",
    "Total_visits_bank",
    "Total_visits_online",
    "Total_calls_made",
]
X = df[feature_columns].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

k_values = range(2, 11)
inertias = []
for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X_scaled)
    inertias.append(model.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(list(k_values), inertias, marker="o")
plt.xticks(list(k_values))
plt.xlabel("Number of clusters (k)")
plt.ylabel("Within-cluster sum of squares (inertia)")
plt.title("K-Means Elbow Method")
plt.grid(alpha=0.3)
plt.show()

optimal_k = 3
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df["KMeans_Cluster"] = kmeans.fit_predict(X_scaled)
print(f"Selected k = {optimal_k}")
print(
    "Rationale: k=3 is the elbow where increasing k begins to produce "
    "smaller reductions in inertia, while keeping the model interpretable."
)

#### Question 3 - K-Means clustering result


In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(
    df["Avg_Credit_Limit"],
    df["Total_Credit_Cards"],
    c=df["KMeans_Cluster"],
    cmap="Set1",
    s=70,
)
plt.title("Customer Segments from K-Means")
plt.xlabel("Average Credit Limit")
plt.ylabel("Total Credit Cards")
plt.colorbar(label="Cluster")
plt.tight_layout()
plt.show()

#### Question 4 - DBSCAN using all usable variables

DBSCAN is fitted to all five standardized usable variables. Standardization is important because the credit-limit column has a much larger numeric scale than the other variables.

In [ ]:
from sklearn.cluster import DBSCAN

dbscan = DBSCAN(eps=0.9, min_samples=5)
df["DBSCAN_Cluster"] = dbscan.fit_predict(X_scaled)

n_clusters_dbscan = len(set(dbscan.labels_)) - (1 if -1 in dbscan.labels_ else 0)
n_noise = int((dbscan.labels_ == -1).sum())
print(f"DBSCAN clusters: {n_clusters_dbscan}")
print(f"Noise observations: {n_noise}")
print(
    "Cluster label -1 represents observations classified as noise rather "
    "than as members of a density-connected cluster."
)

#### Question 5 - DBSCAN k-distance elbow

DBSCAN does not select a number of clusters directly. The elbow in the sorted k-nearest-neighbor distance curve is used to select `eps`; `k` is set to `min_samples`.

In [ ]:
from sklearn.neighbors import NearestNeighbors

min_samples = 5
neighbors = NearestNeighbors(n_neighbors=min_samples)
neighbors.fit(X_scaled)
distances, _ = neighbors.kneighbors(X_scaled)
k_distances = sorted(distances[:, -1])
plt.figure(figsize=(8, 5))
plt.plot(k_distances)
plt.axhline(0.9, color="red", linestyle="--", label="Selected eps = 0.9")
plt.xlabel("Sorted observations")
plt.ylabel(f"Distance to {min_samples}th nearest neighbor")
plt.title("DBSCAN k-Distance Elbow Plot")
plt.legend()
plt.grid(alpha=0.3)
plt.show()
print(
    "Analytical rationale: eps=0.9 is chosen near the sharp rise in the "
    "k-distance curve, separating dense points from likely outliers."
)

#### Question 6 - Silhouette score and Davies-Bouldin index

Noise points (`-1`) are removed before calculating the metrics because they are not a real DBSCAN cluster.

In [ ]:
from sklearn.metrics import davies_bouldin_score, silhouette_score
cluster_labels = dbscan.labels_
non_noise = cluster_labels != -1
evaluation_labels = cluster_labels[non_noise]
evaluation_data = X_scaled[non_noise]
if len(set(evaluation_labels)) >= 2:
    silhouette = silhouette_score(evaluation_data, evaluation_labels)
    dbi = davies_bouldin_score(evaluation_data, evaluation_labels)
    print(f"Silhouette score: {silhouette:.4f}")
    print(f"Davies-Bouldin index: {dbi:.4f}")
    print(
        "Higher silhouette is better, while a lower Davies-Bouldin index "
        "indicates more compact and better-separated clusters."
    )
else:
    print(
        "Metrics cannot be calculated because DBSCAN produced fewer than "
        "two non-noise clusters. Adjust eps or min_samples."
    )